# BNMP — Carga silver

Lê os JSON brutos gravados por `00_extract_bronze.ipynb` no Lakehouse
`mp_bronze` e materializa as tabelas do Warehouse `mp_silver`, usando
`python/src/modulos/bnmp/etl/load_silver.py`.

Tabelas geradas:
- `bnmp_dominio` — as ~90 listas de referência de `/dominios` achatadas
  (status, tipos de peça, motivos de expedição, UFs etc.).
- `bnmp_pessoa_carga` — todas as linhas coletadas, com a consulta e a página
  de origem (pode conter a mesma pessoa mais de uma vez).
- `bnmp_pessoa` — uma linha por pessoa, deduplicada. **É a tabela de uso.**

A deduplicação existe porque a paginação por offset percorre dados vivos: o
total de registros muda durante a coleta e a mesma pessoa pode aparecer em
duas páginas.

**Pré-requisitos:**
- `00_extract_bronze.ipynb` executado, com o manifesto da consulta concluído.
- Identidade do notebook com leitura/escrita no `mp_bronze` (a carga usa uma
  área de staging em `bnmp/staging/`) e escrita no `mp_silver`.

In [ ]:
%pip install --quiet git+https://github.com/mpsp-jurimetria/proj202607.git#subdirectory=python

## Configuração

In [ ]:
import os

os.environ["FABRIC_WORKSPACE_ID"] = "<id do workspace>"
os.environ["FABRIC_LAKEHOUSE_ID"] = "<id do lakehouse mp_bronze>"
os.environ["FABRIC_WAREHOUSE_SILVER_HOST"] = "<host>.datawarehouse.fabric.microsoft.com"
os.environ["FABRIC_WAREHOUSE_SILVER_NAME"] = "mp_silver"

## Execução

Liste em `consultas_pessoas` todas as consultas coletadas no bronze que devem
compor a silver (uma por UF, por exemplo). A carga é completa: as tabelas são
reescritas do zero a cada execução.

In [ ]:
from src.infra.warehouse import get_silver_engine
from src.modulos.bnmp.etl.load_silver import carregar_silver

engine = get_silver_engine()
carregar_silver(engine, consultas_pessoas=["pessoas_uf-26_ativo-1"])

## Verificação

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    coletadas, distintas = conn.execute(text(
        "SELECT COUNT(*), COUNT(DISTINCT pessoa_id_api) FROM bnmp_pessoa_carga"
    )).one()
    print(f"bnmp_pessoa_carga: {coletadas} linhas, {distintas} pessoas distintas")

    total, unicas = conn.execute(text(
        "SELECT COUNT(*), COUNT(DISTINCT pessoa_id_api) FROM bnmp_pessoa"
    )).one()
    print(f"bnmp_pessoa: {total} linhas, {unicas} pessoas distintas (devem ser iguais)")

    dominios = conn.execute(text(
        "SELECT COUNT(*) FROM bnmp_dominio WHERE dominio = 'statusPessoas'"
    )).scalar()
    print(f"bnmp_dominio/statusPessoas: {dominios} itens")

    amostra = conn.execute(text(
        "SELECT TOP 5 nome, sexo_descricao, status_pessoa_descricao, municipio_nome, uf_sigla"
        " FROM bnmp_pessoa"
    )).all()
    for linha in amostra:
        print(linha)